<a href="https://colab.research.google.com/github/davidanbernal/ia/blob/main/actividad4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# APRENDIZAJE NO SUPERVISADO DESDE BASE DE CONOCIMIENTO
# Agrupación de rutas con K-Means
# ==========================================

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans

# ------------------------------------------
# 1. BASE DE CONOCIMIENTO
# ------------------------------------------
red_transporte = {
    "Portal Norte": [("Calle 100", 10), ("Usaquén", 15)],
    "Calle 100": [("Portal Norte", 10), ("Héroes", 8)],
    "Héroes": [("Calle 100", 8), ("Calle 72", 7)],
    "Calle 72": [("Héroes", 7), ("Calle 45", 9), ("Chapinero", 6)],
    "Calle 45": [("Calle 72", 9), ("Av. Jiménez", 10)],
    "Av. Jiménez": [("Calle 45", 10), ("Centro", 5)],
    "Centro": [("Av. Jiménez", 5), ("Museo Nacional", 7)],
    "Museo Nacional": [("Centro", 7)],
    "Chapinero": [("Calle 72", 6)],
    "Usaquén": [("Portal Norte", 15)]
}

# ------------------------------------------
# 2. CONVERTIR LA BASE DE CONOCIMIENTO A DATASET
# ------------------------------------------
datos = []

for origen, conexiones in red_transporte.items():
    for destino, tiempo in conexiones:
        datos.append({
            "origen": origen,
            "destino": destino,
            "tiempo_min": tiempo
        })

df = pd.DataFrame(datos)

print("DATASET GENERADO DESDE LA BASE DE CONOCIMIENTO:")
print(df)

# ------------------------------------------
# 3. PREPROCESAMIENTO
# ------------------------------------------
# origen y destino son texto
# tiempo_min es numérico
preprocesador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), ["origen", "destino"]),
        ("num", StandardScaler(), ["tiempo_min"])
    ]
)

# ------------------------------------------
# 4. MODELO NO SUPERVISADO
# ------------------------------------------
modelo = Pipeline([
    ("prep", preprocesador),
    ("kmeans", KMeans(n_clusters=3, random_state=42, n_init=10))
])

# ------------------------------------------
# 5. ENTRENAR EL MODELO
# ------------------------------------------
modelo.fit(df)

# ------------------------------------------
# 6. ASIGNAR GRUPO A CADA RUTA
# ------------------------------------------
df["grupo"] = modelo.predict(df)

print("\nRUTAS AGRUPADAS:")
print(df)

# ------------------------------------------
# 7. MOSTRAR RESULTADOS POR GRUPO
# ------------------------------------------
print("\nRESUMEN DE GRUPOS:")
for grupo in sorted(df["grupo"].unique()):
    print(f"\nGrupo {grupo}:")
    grupo_df = df[df["grupo"] == grupo]
    print(grupo_df[["origen", "destino", "tiempo_min"]])

# ------------------------------------------
# 8. NUEVA CLASIFICACIÓN
# ------------------------------------------
nueva_ruta = pd.DataFrame([
    {"origen": "Portal Norte", "destino": "Calle 100", "tiempo_min": 10}
])

grupo_nuevo = modelo.predict(nueva_ruta)

print("\nNUEVA RUTA:")
print(nueva_ruta)
print("Esta ruta pertenece al grupo:", grupo_nuevo[0])

DATASET GENERADO DESDE LA BASE DE CONOCIMIENTO:
            origen         destino  tiempo_min
0     Portal Norte       Calle 100          10
1     Portal Norte         Usaquén          15
2        Calle 100    Portal Norte          10
3        Calle 100          Héroes           8
4           Héroes       Calle 100           8
5           Héroes        Calle 72           7
6         Calle 72          Héroes           7
7         Calle 72        Calle 45           9
8         Calle 72       Chapinero           6
9         Calle 45        Calle 72           9
10        Calle 45     Av. Jiménez          10
11     Av. Jiménez        Calle 45          10
12     Av. Jiménez          Centro           5
13          Centro     Av. Jiménez           5
14          Centro  Museo Nacional           7
15  Museo Nacional          Centro           7
16       Chapinero        Calle 72           6
17         Usaquén    Portal Norte          15

RUTAS AGRUPADAS:
            origen         destino  tiemp